# 01 — Bronze: Raw ingestion

Reads the Kaggle `flights_sample_3m.csv` from the raw volume, writes a Delta
table with an ingestion timestamp, and reports row counts. Idempotent: uses a
deterministic full overwrite so a rerun leaves the same row count.

**Run order:** `01_bronze` → `02_eda` → `03_silver` → `04_gold` → `05_train` → `06_api_ingest` → `07_score`.

## Environment
Requires serverless **environment version 4** (needed later for `pyspark.ml` in `05_train`).

In [0]:
import sys
sys.path.append("..")  # so `import src` works from /Workspace Git folder

from pyspark.sql.functions import current_timestamp
from src import config

## Read the source CSV

In [0]:
bronze_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(config.SOURCE_CSV)
    .withColumn("bronze_ingested_at", current_timestamp())
)

bronze_count = bronze_df.count()
print(f"Source rows: {bronze_count:,}")
print(f"Columns:     {len(bronze_df.columns)}")

Source rows: 3,000,000
Columns:     33


## Write Bronze Delta table (overwrite = idempotent)

In [0]:
(
    bronze_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(config.BRONZE)
)

print(f"Wrote {config.BRONZE}")

Wrote workspace.flights.bronze_flights


## Verify

In [0]:
verified = spark.table(config.BRONZE)
print(f"Bronze rows: {verified.count():,}")
verified.printSchema()

Bronze rows: 3,000,000
root
 |-- FL_DATE: date (nullable = true)
 |-- AIRLINE: string (nullable = true)
 |-- AIRLINE_DOT: string (nullable = true)
 |-- AIRLINE_CODE: string (nullable = true)
 |-- DOT_CODE: integer (nullable = true)
 |-- FL_NUMBER: integer (nullable = true)
 |-- ORIGIN: string (nullable = true)
 |-- ORIGIN_CITY: string (nullable = true)
 |-- DEST: string (nullable = true)
 |-- DEST_CITY: string (nullable = true)
 |-- CRS_DEP_TIME: integer (nullable = true)
 |-- DEP_TIME: double (nullable = true)
 |-- DEP_DELAY: double (nullable = true)
 |-- TAXI_OUT: double (nullable = true)
 |-- WHEELS_OFF: double (nullable = true)
 |-- WHEELS_ON: double (nullable = true)
 |-- TAXI_IN: double (nullable = true)
 |-- CRS_ARR_TIME: integer (nullable = true)
 |-- ARR_TIME: double (nullable = true)
 |-- ARR_DELAY: double (nullable = true)
 |-- CANCELLED: double (nullable = true)
 |-- CANCELLATION_CODE: string (nullable = true)
 |-- DIVERTED: double (nullable = true)
 |-- CRS_ELAPSED_TIME: d

In [0]:
%sql
DESCRIBE HISTORY workspace.flights.bronze_flights;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
3,2026-09-10T20:49:22.000Z,71506073265303,omitelkady1@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, canOverwriteSchema -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(1485683378104419),8e0f041e-539a-41ce-904a-ab9663d8060a,0910-204652-6nnmsws0-v2n,2,WriteSerializable,false,"Map(numFiles -> 8, numRemovedFiles -> 8, numRemovedBytes -> 80904607, numDeletionVectorsRemoved -> 0, numOutputRows -> 3000000, numOutputBytes -> 80904607)",null,Databricks-Runtime/19.6.x-aarch64-photon-scala2.13
2,2026-09-09T19:08:35.000Z,71506073265303,omitelkady1@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, canOverwriteSchema -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(1485683378104419),b4a40424-3626-4779-b78a-0da0749635e7,0909-190418-epu4uw4o-v2n,1,WriteSerializable,false,"Map(numFiles -> 8, numRemovedFiles -> 8, numRemovedBytes -> 80904607, numDeletionVectorsRemoved -> 0, numOutputRows -> 3000000, numOutputBytes -> 80904607)",null,Databricks-Runtime/19.6.x-aarch64-photon-scala2.13
1,2026-09-09T18:38:28.000Z,71506073265303,omitelkady1@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, canOverwriteSchema -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(1485683378104419),3ff75334-24ac-4362-9cc1-4dfc5a7ad9e8,0909-183647-gs1dp17m-v2n,0,WriteSerializable,false,"Map(numFiles -> 8, numRemovedFiles -> 8, numRemovedBytes -> 80904615, numDeletionVectorsRemoved -> 0, numOutputRows -> 3000000, numOutputBytes -> 80904607)",null,Databricks-Runtime/19.6.x-aarch64-photon-scala2.13
0,2026-08-30T07:09:56.000Z,71506073265303,omitelkady1@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, canOverwriteSchema -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(1485683378104419),f94006a6-b456-4cf5-8faa-b9afcf641189,0830-070804-aa8kww6x-v2n,null,WriteSerializable,false,"Map(numFiles -> 8, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 3000000, numOutputBytes -> 80904615)",null,Databricks-Runtime/19.2.x-aarch64-photon-scala2.13
